# ELT - NYC Yellow Taxi (Janeiro/2024)

Esqueleto do pipeline. Estrutura sugerida:

1. Setup e conexão
2. Carga da camada raw (dados brutos, sem transformação)
3. Verificação básica da carga
4. Perfilamento dos dados 
5. Camada de tratamento / staging 
6. Modelo dimensional 
7. Camada de consumo analítico 

## 1. Setup e conexão

In [1]:
import duckdb
import os

# Caminhos dos arquivos de origem — ajustem se estiverem em outra pasta
PARQUET_PATH = "yellow_tripdata_2024-01.parquet"
ZONE_LOOKUP_PATH = "taxi_zone_lookup.csv"

# Arquivo de banco persistente em disco (fica salvo entre execuções)
DB_PATH = "nyc_taxi.duckdb"

con = duckdb.connect(DB_PATH)

## 2. Carga da camada raw (dados brutos, sem transformação)

A camada raw deve ser uma cópia fiel do
que chegou, para fins de auditoria e rastreabilidade. 

In [2]:
con.execute(f"""
    CREATE OR REPLACE TABLE raw_yellow_tripdata AS
    SELECT *
    FROM read_parquet('{PARQUET_PATH}')
""")

In [3]:
con.execute(f"""
    CREATE OR REPLACE TABLE raw_taxi_zone_lookup AS
    SELECT *
    FROM read_csv_auto('{ZONE_LOOKUP_PATH}')
""")

## 3. Verificação básica da carga


In [4]:
con.execute("SELECT COUNT(*) AS total_linhas FROM raw_yellow_tripdata").fetchdf()

,total_linhas
0,2964624


In [5]:
con.execute("SELECT * FROM raw_yellow_tripdata LIMIT 5").fetchdf()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.80,1,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.70,1,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.40,1,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.80,1,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0


In [6]:
con.execute("SELECT COUNT(*) AS total_linhas FROM raw_taxi_zone_lookup").fetchdf()

,total_linhas
0,265


In [7]:
con.execute("SELECT * FROM raw_taxi_zone_lookup LIMIT 5").fetchdf()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [18]:
con.execute("DESCRIBE raw_yellow_tripdata").fetchdf()

,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,tpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,tpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,passenger_count,BIGINT,YES,None,None,None
4,trip_distance,DOUBLE,YES,None,None,None
5,RatecodeID,BIGINT,YES,None,None,None
6,store_and_fwd_flag,VARCHAR,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,DOLocationID,INTEGER,YES,None,None,None
9,payment_type,BIGINT,YES,None,None,None


In [19]:
con.execute("DESCRIBE raw_taxi_zone_lookup").fetchdf()

,column_name,column_type,null,key,default,extra
0,LocationID,BIGINT,YES,None,None,None
1,Borough,VARCHAR,YES,None,None,None
2,Zone,VARCHAR,YES,None,None,None
3,service_zone,VARCHAR,YES,None,None,None


## 4. Perfilamento dos dados

Coisas a investigar em `raw_yellow_tripdata` e `raw_taxi_zone_lookup`:
- Nulos por coluna
- Duplicidade de registros
- Cardinalidade e domínio das colunas categóricas (ex.: payment_type, RatecodeID)
- Valores improváveis (datas fora de janeiro/2024, distâncias/valores negativos, etc.)
- Correspondência dos códigos de localização entre os dois arquivos

Guardem aqui as consultas usadas e o que cada uma revelou.

# TODO: Perfilamento

### 4.1 Valores Nulos

In [8]:
con.execute("""
    SELECT COUNT(*) AS total_linhas,
           COUNT(*) - COUNT(VendorID) AS nulos_vendorid,
           COUNT(*) - COUNT(tpep_pickup_datetime) AS nulos_pickup_datetime,
           COUNT(*) - COUNT(tpep_dropoff_datetime) AS nulos_dropoff_datetime,
           COUNT(*) - COUNT(passenger_count) AS nulos_passenger_count,
           COUNT(*) - COUNT(trip_distance) AS nulos_trip_distance,
           COUNT(*) - COUNT(RatecodeID) AS nulos_ratecodeid,
           COUNT(*) - COUNT(store_and_fwd_flag) AS nulos_store_and_fwd_flag,
           COUNT(*) - COUNT(PULocationID) AS nulos_pulocationid,
           COUNT(*) - COUNT(DOLocationID) AS nulos_dolocationid,
           COUNT(*) - COUNT(payment_type) AS nulos_payment_type,
           COUNT(*) - COUNT(fare_amount) AS nulos_fare_amount,
           COUNT(*) - COUNT(extra) AS nulos_extra,
           COUNT(*) - COUNT(mta_tax) AS nulos_mta_tax,
           COUNT(*) - COUNT(tip_amount) AS nulos_tip_amount,
           COUNT(*) - COUNT(tolls_amount) AS nulos_tolls_amount,
           COUNT(*) - COUNT(improvement_surcharge) AS nulos_improvement_surcharge,
           COUNT(*) - COUNT(total_amount) AS nulos_total_amount,
           COUNT(*) - COUNT(congestion_surcharge) AS nulos_congestion_surcharge,
           COUNT(*) - COUNT(Airport_fee) AS nulos_airport_fee
    FROM raw_yellow_tripdata
""").fetchdf()

,total_linhas,nulos_vendorid,nulos_pickup_datetime,nulos_dropoff_datetime,nulos_passenger_count,nulos_trip_distance,nulos_ratecodeid,nulos_store_and_fwd_flag,nulos_pulocationid,nulos_dolocationid,nulos_payment_type,nulos_fare_amount,nulos_extra,nulos_mta_tax,nulos_tip_amount,nulos_tolls_amount,nulos_improvement_surcharge,nulos_total_amount,nulos_congestion_surcharge,nulos_airport_fee
0,2964624,0,0,0,140162,0,140162,140162,0,0,0,0,0,0,0,0,0,0,140162,140162


### 4.2 Linhas Duplicadas

In [9]:
con.execute("""
    SELECT VendorID, tpep_pickup_datetime, tpep_dropoff_datetime,
           PULocationID, DOLocationID, COUNT(*) AS qtd
    FROM raw_yellow_tripdata
    GROUP BY VendorID, tpep_pickup_datetime, tpep_dropoff_datetime, PULocationID, DOLocationID
    HAVING COUNT(*) > 1
    ORDER BY qtd DESC
""").fetchdf()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,qtd
0,2,2024-01-13 07:33:00,2024-01-13 07:35:03,236,75,2
1,2,2024-01-13 08:37:46,2024-01-13 08:54:58,249,97,2
2,2,2024-01-13 10:56:42,2024-01-13 11:13:36,237,211,2
3,2,2024-01-13 11:41:26,2024-01-13 11:45:56,142,151,2
4,2,2024-01-13 12:24:23,2024-01-13 12:38:36,48,113,2
...,...,...,...,...,...,...
31104,2,2024-01-20 22:09:35,2024-01-20 22:12:30,236,75,2
31105,2,2024-01-20 22:25:39,2024-01-20 23:03:59,161,265,2
31106,2,2024-01-20 22:45:29,2024-01-20 23:03:09,239,170,2
31107,2,2024-01-20 22:52:23,2024-01-20 22:57:41,230,186,2


### 4.3 Payment_type

In [10]:
con.execute("""
    SELECT payment_type, COUNT(*) AS qtd
    FROM raw_yellow_tripdata
    GROUP BY payment_type
    ORDER BY qtd DESC
""").fetchdf()

,payment_type,qtd
0,1,2319046
1,2,439191
2,0,140162
3,4,46628
4,3,19597


### 4.4 RatecodeID

In [11]:
con.execute("""
    SELECT RatecodeID, COUNT(*) AS qtd
    FROM raw_yellow_tripdata
    GROUP BY RatecodeID
    ORDER BY qtd DESC
""").fetchdf()

,RatecodeID,qtd
0,1,2663350
1,<NA>,140162
2,2,98713
3,99,28663
4,5,19410
5,3,7954
6,4,6365
7,6,7


### 4.5 VendorID

In [12]:
con.execute("""
    SELECT VendorID, COUNT(*) AS qtd
    FROM raw_yellow_tripdata
    GROUP BY VendorID
    ORDER BY qtd DESC
""").fetchdf()

,VendorID,qtd
0,2,2234632
1,1,729732
2,6,260


### 4.6 Passenger_count

In [13]:
con.execute("""
    SELECT passenger_count, COUNT(*) AS qtd
    FROM raw_yellow_tripdata
    GROUP BY passenger_count
    ORDER BY passenger_count
""").fetchdf()

,passenger_count,qtd
0,0,31465
1,1,2188739
2,2,405103
3,3,91262
4,4,51974
5,5,33506
6,6,22353
7,7,8
8,8,51
9,9,1


### 4.7 store_and_fwd_flag

In [14]:
con.execute("""
    SELECT store_and_fwd_flag, COUNT(*) AS qtd
    FROM raw_yellow_tripdata
    GROUP BY store_and_fwd_flag
    ORDER BY qtd DESC
""").fetchdf()

,store_and_fwd_flag,qtd
0,N,2813126
1,NaN,140162
2,Y,11336


## 5. Camada de tratamento (staging)

*( cada transformação deve ter justificativa
baseada no que foi encontrado no perfilamento acima.)*

In [15]:
# TODO: tratamento / staging

## 6. Modelo dimensional

*( declarem aqui a granularidade da fato antes
de criar qualquer tabela.)*

- Grão da fato:
- Dimensões planejadas:

In [16]:
# TODO: tabela fato e dimensões

## 7. Camada de consumo analítico

*(consultas de negócio sobre tempo, geografia,
financeiro e operação, sempre a partir do modelo dimensional.)*

In [17]:
# TODO: consultas analíticas